# Social Networks BO Plot

Plot regret against BO iteration from the latest social-network BO sweep.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path("results")
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

csv_candidates = sorted(RESULTS_DIR.glob("social_networks_bo_sweep_*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No sweep results found in {RESULTS_DIR}")

csv_path = csv_candidates[-1]
df = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
df.head()

In [ ]:
algorithm_order = ["GRF+Thompson", "BFS", "DFS", "RandomSearch"]
algorithm_colors = {
    "GRF+Thompson": "#1f77b4",
    "BFS": "#2ca02c",
    "DFS": "#d95f02",
    "RandomSearch": "#e6a000",
}
dataset_order = [
    dataset
    for dataset in ["enron", "facebook", "twitch", "youtube"]
    if dataset in df["dataset"].unique()
]

stats = (
    df.groupby(["dataset", "algorithm", "step"])["regret"]
    .agg(["mean", "std"])
    .reset_index()
)

n_datasets = len(dataset_order)
n_cols = 2 if n_datasets > 1 else 1
n_rows = math.ceil(n_datasets / n_cols)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(6 * n_cols, 4 * n_rows),
    squeeze=False,
    sharex=False,
    sharey=False,
)

for ax, dataset_name in zip(axes.flat, dataset_order):
    dataset_stats = stats[stats["dataset"] == dataset_name]
    for algorithm in algorithm_order:
        algorithm_stats = dataset_stats[dataset_stats["algorithm"] == algorithm]
        if algorithm_stats.empty:
            continue
        x = algorithm_stats["step"].to_numpy()
        mean = algorithm_stats["mean"].to_numpy()
        std = algorithm_stats["std"].fillna(0.0).to_numpy()
        color = algorithm_colors[algorithm]
        ax.plot(x, mean, label=algorithm, color=color, linewidth=2)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.2)

    ax.set_title(dataset_name.replace("_", " ").title())
    ax.set_xlabel("BO iteration")
    ax.set_ylabel("Regret")
    ax.grid(True, alpha=0.3)

for ax in axes.flat[n_datasets:]:
    ax.axis("off")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", frameon=False)
fig.tight_layout(rect=(0, 0, 0.88, 1))

png_path = PLOTS_DIR / "social_networks_bo_regret.png"
pdf_path = PLOTS_DIR / "social_networks_bo_regret.pdf"
fig.savefig(png_path, dpi=200, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
plt.show()